In [ ]:
from pathlib import Path
import pickle as pkl
import json as json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
path = os.path.abspath("/path/to/code/labelnoise-on-UQ")
os.environ["PYTHONPATH"] = path + os.pathsep + os.environ.get("PYTHONPATH", "")
sys.path.insert(0, path)
from src.Utils.vocabulary import (
    TRAIN, TEST, MULTI_TARGET, HARD_TARGET, INDEX, LABELS, PREDICTED, AU, EU, TU,
    index2target_PTB, index2multitarget_ESR
)

%load_ext autoreload
%autoreload 2

# Quantification functions

In [ ]:
def compare_value_to_bin(value, bin_edges):
    """
    Compare a value to a couple of bin edges and return the distance between the value and the closest bin edge.
    
    Parameters:
    - value: The value to compare.
    - bin_edges: A pair of bin edges.
    
    Returns:
    - The distance between the value and the closest bin edge.
    """
    lower_edge, upper_edge = bin_edges
    if value < lower_edge:
        distance = value - lower_edge
    elif value > upper_edge:
        distance = value - upper_edge
    else:
        distance = 0  # The value is within the bin edges
    return distance

def compare_dist_value_mode(exp_bin_values, test_theo, bin_edges=None):
    if bin_edges is None:
        print("Warning: bin_edges is None, using indices as bin edges.")
        bin_edges = np.linspace(0, len(exp_bin_values), len(exp_bin_values) + 1)
    mode_value = np.argmax(exp_bin_values)
    distance = compare_value_to_bin(test_theo, (bin_edges[mode_value], bin_edges[mode_value + 1]))
    return distance / (bin_edges[-1] - bin_edges[0])  # Normalize the distance by the range of bin edges

def compare_dist_value_mean(exp_bin_values, test_theo, bin_edges=None):
    if bin_edges is None:
        print("Warning: bin_edges is None, using indices as bin edges.")
        bin_edges = np.linspace(0, len(exp_bin_values), len(exp_bin_values) + 1)
    bin_weights = (bin_edges[:-1] + bin_edges[1:]) / 2  # Calculate the midpoints of the bins
    mean_value = np.sum(exp_bin_values * bin_weights) / np.sum(exp_bin_values)  # Calculate the mean value based on bin weights
    mean_index = np.argmin(np.abs(bin_edges[:-1] - mean_value))
    # Find the closest bin index to the mean value
    distance = compare_value_to_bin(test_theo, (bin_edges[mean_index], bin_edges[mean_index + 1]))
    return distance / (bin_edges[-1] - bin_edges[0])  # Normalize the distance by the range of bin edges

def compare_dist_value_median(exp_bin_values, test_theo, bin_edges=None):
    if bin_edges is None:
        print("Warning: bin_edges is None, using indices as bin edges.")
        bin_edges = np.linspace(0, len(exp_bin_values), len(exp_bin_values) + 1)
    cumulative_sum = np.cumsum(exp_bin_values)
    median_index = np.searchsorted(cumulative_sum, cumulative_sum[-1] / 2)
    distance = compare_value_to_bin(test_theo, (bin_edges[median_index], bin_edges[median_index + 1]))
    return distance / (bin_edges[-1] - bin_edges[0])  # Normalize the distance by the range of bin edges

def compare_dist_value_closest_global_max(exp_bin_values, test_theo, bin_edges=None):
    if bin_edges is None:
        print("Warning: bin_edges is None, using indices as bin edges.")
        bin_edges = np.linspace(0, len(exp_bin_values), len(exp_bin_values) + 1)
    peak_values = np.where(exp_bin_values == np.max(exp_bin_values))[0] # indices of the global maximum values
    peak_edges = [(bin_edges[p], bin_edges[p + 1]) for p in peak_values] # corresponding bin edges for the global maximum values
    distance_to_edges = [compare_value_to_bin(test_theo, edges) for edges in peak_edges] # distances to the bin edges of the global maximum values
    index_of_closest_dist = np.argmin(np.abs(distance_to_edges)) # index of the closest global maximum to the theoretical value
    return distance_to_edges[index_of_closest_dist] / (bin_edges[-1] - bin_edges[0])  # Normalize the distance by the range of bin edges

def get_closest_local_max(exp_bin_values, test_theo, bin_edges):
    assert bin_edges is not None, "bin_edges must be provided to get_closest_local_max function."
    if bin_edges is None:
        print("Warning: bin_edges is None, using indices as bin edges.")
        bin_edges = np.linspace(0, len(exp_bin_values), len(exp_bin_values) + 1)
    # --- Get all local maxima (including plateaus and border values) ---
    # Search for local maxima
    local_maxima = (np.diff(np.sign(np.diff(exp_bin_values))) < 0).nonzero()[0] + 1  # indices of local maxima
    
    # Get plateaus (consecutive indices with the same value) and add them to local maxima
    plateaus = np.where(np.diff(exp_bin_values) == 0)[0] + 1
    for p in plateaus:
        if (p - 1 in local_maxima or p + 1 in local_maxima) and p not in local_maxima:
            local_maxima = np.append(local_maxima, p)
    
    # Check if value at the borders are local maxima
    if exp_bin_values[-1] > exp_bin_values[-2] and (len(exp_bin_values) - 1) not in local_maxima:
        local_maxima = np.append(local_maxima, len(exp_bin_values) - 1)
    if exp_bin_values[0] > exp_bin_values[1] and 0 not in local_maxima:
        local_maxima = np.append(local_maxima, 0)
    
    if len(local_maxima) == 0:
        return None, None, None  # No local maxima found
    
    # Get the corresponding bin edges for the local maxima and distance to theoretical value
    local_maxima_edges = [(bin_edges[l], bin_edges[l + 1]) for l in local_maxima]
    distance_to_edges = [compare_value_to_bin(test_theo, edges) for edges in local_maxima_edges] # distances to the bin edges of the local maximum values
    index_of_closest_dist = np.argmin(np.abs(distance_to_edges)) # index of the closest local maximum to the theoretical value
    
    closest_local_max = local_maxima[index_of_closest_dist]
    distance_to_closest_local_max = distance_to_edges[index_of_closest_dist]
    return closest_local_max, distance_to_closest_local_max, local_maxima

def compare_dist_value_closest_local_max(exp_bin_values, test_theo, bin_edges=None, print_local_maxima=False):
    if bin_edges is None:
        print("Warning: bin_edges is None, using indices as bin edges.")
        bin_edges = np.linspace(0, len(exp_bin_values), len(exp_bin_values) + 1)
    # --- Get all local maxima (including plateaus and border values) ---
    closest_local_max, distance_to_closest_local_max, local_maxima = get_closest_local_max(exp_bin_values, test_theo, bin_edges)
    if closest_local_max is None:
        return None  # No local maxima found
    if print_local_maxima:
        print(f"Local maxima indices: {local_maxima}, values: {exp_bin_values[local_maxima]}")
        print(f"Closest local maximum index: {closest_local_max}, value: {exp_bin_values[closest_local_max]}, distance to theoretical value: {distance_to_closest_local_max}")
    print(f"Theoretical mean value: {test_theo}, closest local maximum bins: [{bin_edges[closest_local_max]}, {bin_edges[closest_local_max + 1]}], distance to theoretical value: {distance_to_closest_local_max}")
    return distance_to_closest_local_max / (bin_edges[-1] - bin_edges[0])  # Normalize the distance by the range of bin edges

def compare_dist_value_two_first_max(exp_bin_values, test_theo, bin_edges=None):
    if bin_edges is None:
        print("Warning: bin_edges is None, using indices as bin edges.")
        bin_edges = np.linspace(0, len(exp_bin_values), len(exp_bin_values) + 1)
    # --- Get all local maxima (including plateaus and border values) ---
    # Search for local maxima
    local_maxima = (np.diff(np.sign(np.diff(exp_bin_values))) < 0).nonzero()[0] + 1  # indices of local maxima
    
    # Check if value at the borders are local maxima
    if exp_bin_values[-1] > exp_bin_values[-2] and (len(exp_bin_values) - 1) not in local_maxima:
        local_maxima = np.append(local_maxima, len(exp_bin_values) - 1)
    if exp_bin_values[0] > exp_bin_values[1] and 0 not in local_maxima:
        local_maxima = np.append(local_maxima, 0)
    
    if len(local_maxima) == 0:
        return None  # No local maxima found
    
    # Sort maxima by value and get two first local maxima
    print(f"Local maxima indices: {local_maxima}, values: {exp_bin_values[local_maxima]}")
    local_maxima = sorted(local_maxima, key=lambda x: exp_bin_values[x], reverse=True)
    print(f"Sorted local maxima indices: {local_maxima}, values: {exp_bin_values[local_maxima]}")
    local_maxima = local_maxima[:2]
    
    # Get plateaus (consecutive indices with the same value) and add them to local maxima, add them after sorting to ensure we get the two first local maxima
    plateaus = np.where(np.diff(exp_bin_values) == 0)[0] + 1
    for p in plateaus:
        if (p - 1 in local_maxima or p + 1 in local_maxima) and p not in local_maxima:
            local_maxima = np.append(local_maxima, p)

    # Get the corresponding bin edges for the two first local maxima and distance to theoretical value
    local_maxima_edges = [(bin_edges[l], bin_edges[l + 1]) for l in local_maxima]
    distance_to_edges = [compare_value_to_bin(test_theo, edges) for edges in local_maxima_edges] # distances to the bin edges of the first two local maximum values
    index_of_closest_dist = np.argmin(np.abs(distance_to_edges)) # index of the closest of the two first local maxima to the theoretical value
    return distance_to_edges[index_of_closest_dist] / (bin_edges[-1] - bin_edges[0])  # Normalize the distance by the range of bin edges

compare_distance_functions = {
    "mode": compare_dist_value_mode,
    "mean": compare_dist_value_mean,
    "median": compare_dist_value_median,
    "closest global max": compare_dist_value_closest_global_max,
    "closest local max": compare_dist_value_closest_local_max,
    "two first local max": compare_dist_value_two_first_max
}

## Test with simple distributions

In [ ]:
l = [0, 2, 4, 3, 0, 5, 5, 3, 5, 5, 5, 1, 5]
b = np.linspace(0, len(l), len(l) + 1)
local_maxima = (np.diff(np.sign(np.diff(l))) < 0).nonzero()[0] + 1  # indices of local maxima
print(f"Local maxima indices: {local_maxima}, values: {np.array(l)[local_maxima]}")
local_maxima_edges = [(b[l], b[l + 1]) for l in local_maxima]
print(b)
print(f"Local maxima edges: {local_maxima_edges}")

In [ ]:
l = [0, 2, 4, 3, 0, 5, 5, 3, 5, 5, 5, 1, 5]
dp = np.diff(l) # [ 2  2 -1 -3  5  0 -2  2  0  0 -4  4]
print(dp)
# --> Local max are the index of the number which has a change in index (or 0 if plateau)
print(np.sign(dp))
print(np.diff(np.sign(dp)))


In [ ]:
tests = [
    {
        "name": "bell",
        "exp": [0, 0, 3, 6, 8, 9, 8, 6, 3, 0]
    },
    {
        "name": "asym bell",
        "exp": [0, 0, 2, 3, 5, 9, 8, 6, 3, 0]
    },
    {
        "name": "plate bell",
        "exp": [0, 0, 2, 3, 9, 9, 9, 6, 3, 0]
    },
    {
        "name": "hills",
        "exp": [0, 3, 8, 9, 8, 3, 4, 5, 3, 1]
    },
    {
        "name": "mirrored bell",
        "exp": [3, 8, 9, 8, 3, 8, 9, 8, 3, 0]
    },
    {
        "name": "increasing",
        "exp": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
    },
    {
        "name": "decreasing",
        "exp": [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]
    },
    # {
    #     "name": "flat",
    #     "exp": [5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
    # },
    {
        "name": "real",
        "exp": [0, 5, 4, 5, 4, 2, 3, 4, 6, 2]
    }
]
test_bin_edges = np.linspace(0, 1, 11)  # 10 bins from 0 to 1

In [ ]:
tested_fn = [("mode", compare_distance_functions["mode"]), 
             ("mean", compare_distance_functions["mean"]),
             ("median", compare_distance_functions["median"]),
             ("closest global max", compare_distance_functions["closest global max"]),
             ("closest local max", lambda exp, theo, bin_edges: compare_distance_functions["closest local max"](exp, theo, bin_edges=bin_edges, print_local_maxima=True)),
             ("two first local max", compare_distance_functions["two first local max"])
             ]
test_theo_list = np.linspace(test_bin_edges[0], test_bin_edges[-1], 100)  # N theoretical values from the first to the last bin edge
markers = ["+", "x", "1", "2", "3", "4", "s", "p", "*", "h", "H", "+", "x", "D", "|", "_"]
for test_exp in tests:
    test_name = test_exp["name"]
    test_exp_values = np.array(test_exp["exp"])
    
    # Show distribution
    plt.figure(figsize=(8, 4))
    # plt.subplot(1, 2, 1)
    ax1 = plt.gca()
    bin_width = test_bin_edges[1] - test_bin_edges[0]
    ax1.bar(test_bin_edges[:-1] + bin_width / 2, test_exp_values, width=bin_width, color='gray', alpha=0.5)
    ax1.set_ylabel("Experimental values")
    ax1.set_xlabel("Théoretical position")
    plt.title(f"Distribution for {test_name} test")
    plt.xticks(test_bin_edges)
    
    # Show quantification for each function
    # plt.subplot(1, 2, 2)
    ax2 = ax1.twinx()  # Create a second y-axis sharing the same x-axis
    for i, (fn_name, fn) in enumerate(tested_fn):
        quantif = [fn(test_exp_values, test_theo, bin_edges=test_bin_edges) for test_theo in test_theo_list]
        ax2.plot(test_theo_list, quantif, marker=markers[i], label=f"Fn {fn_name}")
        ax2.plot(test_theo_list, np.abs(quantif), '--', color=ax2.lines[-1].get_color())
        # ax2.plot(test_theo_list, np.abs(quantif), marker=markers[i], label=f"Fn {fn_name}")
    ax2.plot([test_bin_edges[0], test_bin_edges[-1]], [0, 0], color='black', linestyle='--', label="Zero line")
    ax2.set_ylabel("Distance")
    ax2.set_xlabel("Potential values")
    plt.xticks(test_bin_edges)
    plt.grid()
    # Legend outside the plot
    plt.legend(bbox_to_anchor=(1.1, 1), loc='upper left')  # Place legend outside the plot
    plt.show()


# Use function between theo and exp

In [ ]:
# General constants
dataset_idx2class = {"MNIST": {k: f"Nb {k}" for k in range(10)}, "ESR": index2multitarget_ESR, "PTB": index2target_PTB}
dataset_colors = {"MNIST": "green", "ESR": "orange", "PTB": "blue"}
nr_colors = {0.: '#93c47dff', 0.2: '#ffd966ff', 0.4: '#e69138ff', 0.6:'#cc0000ff'}
target_names = {"MNIST": HARD_TARGET, "ESR 1DCNNTransf": MULTI_TARGET, "ESR 2DCNN": MULTI_TARGET, "PTB": HARD_TARGET,
                "MNIST same noise": HARD_TARGET, "ESR 1DCNNTransf same noise": MULTI_TARGET}
target_K = {"MNIST": 10, "ESR 1DCNNTransf": 5, "ESR 2DCNN": 5, "PTB": 2}

axis_fontsize = 14
title_fontsize = 16
legend_fontsize = 12

UQ_COLORS = {TU: 'skyblue', AU: 'lightgreen', EU: 'mistyrose'}
UQ_NAMES = [TU, AU, EU]
UQ_NAMES_SHORT = {TU: 'TU', AU: 'AU', EU: 'EU'}
MODES = [TRAIN, TEST]


def aggregate_prediction(pred):
    # pred: (n_classes, n_inference)
    mean_logits = np.mean(pred, axis=1)  # shape (n_classes,)
    return np.argmax(mean_logits)        # final predicted class

loss_viz_names = {"CE_nodropout": "CE", "CE_dropout": "CE + Dropout", "GCE_q0-4_nodropout": "GCE (q=0.4)", "GCE_q0-7_nodropout": "GCE (q=0.7)", "GCE_q1_nodropout": "GCE (q=1.0)",
                 "balancedCE_nodropout": "Balanced CE", "balancedCE_dropout": "Balanced CE + Dropout", "balancedGCE_q0-4_nodropout": " Balanced GCE (q=0.4)", "balancedGCE_q0-7_nodropout": "Balanced GCE (q=0.7)", "balancedGCE_q1_nodropout": "Balanced GCE (q=1.0)",
            }


## Gather paths

In [ ]:
# Gather paths

exp_root_path = Path("/path/to/experiments/publi_UQandNoise/Exp1")
assert exp_root_path.exists(), f"Experiment root path {exp_root_path} does not exist. Please check the path."
exp_summary = {
    "MNIST": [
        "CE_dropout", "CE_nodropout", 
        "GCE_q0-4_nodropout", "GCE_q0-7_nodropout", "GCE_q1_nodropout",
        "RCE_A-16_nodropout", "MAE_nodropout"],
    "ESR 1DCNNTransf": ["balancedCE_dropout", "balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedGCE_q0-7_nodropout", "balancedGCE_q1_nodropout"],
    "ESR 2DCNN": ["balancedCE_dropout", "balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedGCE_q0-7_nodropout", "balancedGCE_q1_nodropout"],
    "PTB": ["CE_dropout", "CE_nodropout", "GCE_q0-4_nodropout", "GCE_q0-7_nodropout", "GCE_q1_nodropout"]
}
exp_roots = {
    "MNIST": ("MNIST", "Exp_ClassifMNIST_2DCNN_"),
    "ESR 1DCNNTransf": ("ESR_1DCNNTransf", "Exp_ClassifESRmulti_1DCNNTransf_"),
    "ESR 2DCNN": ("ESR_2DCNN", "Exp_ClassifESRmulti_2DCNN_"),
    "PTB": ("PTB", "Exp_ClassifPTBsub_1DCNNTransf_")
}
exp_paths = {
    d_name : {
        l_name : {
            nr : exp_root_path/f"{exp_roots[d_name][0]}"/f"{exp_roots[d_name][1]}{l_name}"/f"noise_rate_{str(nr).replace('.', '-')}" for nr in [0., 0.2, 0.4, 0.6]
            } for l_name in exp_summary[d_name]
        } for d_name in exp_summary.keys()
}

nb_runs_per_exp = 10
run_paths = {
    d: 
        {l: 
            {nr: [exp_paths[d][l][nr]/f"run-{i}" for i in range(nb_runs_per_exp) if (exp_paths[d][l][nr]/f"run-{i}").exists()]
             for nr in exp_paths[d][l].keys()
             } 
            for l in exp_paths[d].keys()} 
        for d in exp_paths.keys()
    }

# dataset_names = ["MNIST"]
dataset_names = ["MNIST", "ESR 1DCNNTransf", "ESR 2DCNN", "PTB"]
# dataset_names = ["ESR 1DCNNTransf", "ESR 2DCNN"]
n_dataset = len(dataset_names)
nr_rates = [0., 0.2, 0.4, 0.6]
modes = [TRAIN, TEST]
n_nr = len(nr_rates)
n_modes = len(modes)

#! Predictions and metric without several inference with dropout
pred_name = "predictions.pkl"
metric_name = "metric.json"

included_losses = {
    "MNIST": [
            "CE_nodropout", 
            "GCE_q0-4_nodropout", "GCE_q0-7_nodropout", "GCE_q1_nodropout", 
            # "RCE_A-16_nodropout", 
            # "MAE_nodropout", 
        ],
    "ESR 1DCNNTransf": [
            "balancedCE_nodropout", 
            "balancedGCE_q0-4_nodropout", "balancedGCE_q0-7_nodropout", "balancedGCE_q1_nodropout", 
            # "balancedRCE_A-16_nodropout", 
            # "balancedMAE_nodropout"
        ],
    "ESR 2DCNN": [
            "balancedCE_nodropout", 
            "balancedGCE_q0-4_nodropout", "balancedGCE_q0-7_nodropout", "balancedGCE_q1_nodropout",
            # "balancedRCE_A-16_nodropout", 
            # "balancedMAE_nodropout"
        ],
    "PTB": [
        "CE_nodropout", 
        "GCE_q0-4_nodropout", "GCE_q0-7_nodropout", "GCE_q1_nodropout", 
        # "RCE_A-16_nodropout"
    ]
}


In [ ]:
# Print number of runs per dataset, loss and noise rate
for d_name in dataset_names:
    for l_name in exp_paths[d_name].keys():
        for nr in exp_paths[d_name][l_name].keys():
            print(f"{d_name} - {l_name} - nr {nr}: {len(run_paths[d_name][l_name][nr])} runs")

## Load predictions and metrics

In [ ]:
# Load predictions
predictions = {}
for d in dataset_names:
    predictions[d] = {}
    for l in included_losses[d]:
        predictions[d][l] = {}
        for nr in nr_rates:
            predictions[d][l][nr] = []
            target_key = target_names[d]
            for run_i in run_paths[d][l][nr]:
                with (run_i/pred_name).open("rb") as f:
                    pred = pkl.load(f)
                predictions[d][l][nr].append(pred)

# 
# 40s all

In [ ]:
# Load metrics
# Dict description: metrics[dataset_name][noise_rate][loss_name] = list of metrics for each run
metrics = {}
for d in dataset_names:
    metrics[d] = {}
    for nr in nr_rates:
        metrics[d][nr] = {}
        for l in included_losses[d]:
            target_key = target_names[d]
            metrics[d][nr][l] = []
            for run_i in run_paths[d][l][nr]:
                with (run_i/metric_name).open("r") as f:
                    metric = json.load(f)
                metrics[d][nr][l].append(metric)

# Average metrics over runs
# Dict description: avg_metrics[dataset_name][noise_rate][loss_name] = dict of average metrics over runs
avg_metrics = {}
std_metrics = {}
for d in dataset_names:
    avg_metrics[d] = {}
    std_metrics[d] = {}
    for nr in nr_rates:
        avg_metrics[d][nr] = {}
        std_metrics[d][nr] = {}
        for l in included_losses[d]:
            target_key = target_names[d]
            # Average metrics over runs
            avg_metrics[d][nr][l] = {}
            std_metrics[d][nr][l] = {}
            print(len(metrics[d][nr][l]), "runs for", d, l, "with noise rate", nr)
            for mode in metrics[d][nr][l][0].keys():
                avg_metrics[d][nr][l][mode] = {}
                std_metrics[d][nr][l][mode] = {}
                for m in metrics[d][nr][l][0][mode].keys():
                    if type(metrics[d][nr][l][0][mode][m]) in [int, float]:
                        print(m, "int/float", mode)
                        avg_metrics[d][nr][l][mode][m] = np.mean([metrics[d][nr][l][i][mode][m] for i in range(len(metrics[d][nr][l]))])
                        std_metrics[d][nr][l][mode][m] = np.std([metrics[d][nr][l][i][mode][m] for i in range(len(metrics[d][nr][l]))])
                    else:
                        print(m, "None/0", mode)
                        avg_metrics[d][nr][l][mode][m] = None
                        std_metrics[d][nr][l][mode][m] = None

## Theoretical functions

In [ ]:
# Theoretical minimal top score for each loss and each noise rate (for a given K)
from pathlib import Path


def v_min_CE(eta, K) -> float or np.ndarray:
    return 1 - eta

def v_min_MAE(eta, K) -> float or np.ndarray:
    return 1 *(eta <= (K - 1)/K) # if eta <= (K - 1)/K, v_min = 0, else v_min = 1 (boolean to keep vectorization)

def v_min_GCE(eta, K, q) -> float or np.ndarray:
    if q == 1:
        return 1
    return 1 / (1 + (K - 1) ** (-q / (1 - q)) * (eta / (1 - eta)) ** (1 / (1 - q)))
    
def loss_theo_min(loss_name):
    if "GCE" in loss_name:
        if "q0-4" in loss_name:
            return lambda eta, K: v_min_GCE(eta, K, q=0.4)
        elif "q0-7" in loss_name:
            return lambda eta, K: v_min_GCE(eta, K, q=0.7)
        elif "q1" in loss_name:
            return lambda eta, K: v_min_GCE(eta, K, q=1)
    # if "balanced_geoJSD" in loss_name:
    #     return lambda eta, K: None
    elif "RCE" in loss_name:
        return lambda eta, K: 1. * (eta <= (K - 1)/K) # if eta <= (K - 1)/K, v_min = 0, else v_min = 1 (boolean to keep vectorization)
    elif "MAE" in loss_name:
        return v_min_MAE
    elif "CE" in loss_name:
        return v_min_CE
    print(f"No theoretical minimum defined for loss {loss_name}")
    return lambda eta, K: None


## Prepare different score functions
- Mean of max scores: top_scores_mean
- Histogram concatenated (aggregated): top_scores_concatenated
- Max of mean score: top_scores_max_of_mean

In [ ]:
mode = TRAIN

# Get run top scores
byrun_scores_ordered = {}
for d_name in dataset_names:
    loss_names = included_losses[d_name]
    byrun_scores_ordered[d_name] = {}
    # Get the scores list for each loss and each nr
    for i_l, l_name in enumerate(loss_names):
        byrun_scores_ordered[d_name][l_name] = {}
        for nr in nr_rates:
            byrun_scores_ordered[d_name][l_name][nr] = []
            predictions_nr_l = predictions[d_name][l_name][nr]
            for i_run in range(len(predictions_nr_l)):
                # Initial shape is (n_samples, n_classes, n_inference), with n_inference = 1 in our case
                # Sort the predictions (and labels and index for coherence) according to index to match between runs
                sample_idx = predictions_nr_l[i_run][mode][INDEX]
                predictions_nr_l[i_run][mode][INDEX] = predictions_nr_l[i_run][mode][INDEX][np.argsort(sample_idx)]
                predictions_nr_l[i_run][mode][target_names[d_name]][LABELS] = predictions_nr_l[i_run][mode][target_names[d_name]][LABELS][np.argsort(sample_idx)]
                predictions_nr_l[i_run][mode][target_names[d_name]][PREDICTED] = predictions_nr_l[i_run][mode][target_names[d_name]][PREDICTED][np.argsort(sample_idx)]
                
                # Mean across the different inferences (final shape (n_samples, n_classes))
                run_top_scores = np.mean(predictions_nr_l[i_run][mode][target_names[d_name]][PREDICTED], axis=2)

                byrun_scores_ordered[d_name][l_name][nr].append(run_top_scores)  # shape (n_samples,)


In [ ]:
# Mean of max scores
top_scores_mean = {}
for d_name in dataset_names:
    loss_names = included_losses[d_name]
    top_scores_mean[d_name] = {}
    
    # Get the scores list for each loss and each nr
    for i_l, l_name in enumerate(loss_names):
        top_scores_mean[d_name][l_name] = {}
        for nr in nr_rates:
            top_scores_mean[d_name][l_name][nr] = []
            for i_run in range(len(predictions[d_name][l_name][nr])):
                # Initial shape is (n_samples, n_classes)
                byrun_score = byrun_scores_ordered[d_name][l_name][nr][i_run]  # shape (n_samples, n_classes)
                # Top score across the classes (final shape (n_samples,))
                run_top_scores = np.max(byrun_score, axis=1)  # shape (n_samples,)
                top_scores_mean[d_name][l_name][nr].append(run_top_scores)
            # Mean across the runs (final shape (n_samples,)), average can be done because samples are sorted according to index and thus correspond between runs
            top_scores_mean[d_name][l_name][nr] = np.mean(top_scores_mean[d_name][l_name][nr], axis=0)

In [ ]:
# Max score concat
top_scores_concat = {}
for d_name in dataset_names:
    loss_names = included_losses[d_name]
    top_scores_concat[d_name] = {}
    
    # Get the scores list for each loss and each nr
    for i_l, l_name in enumerate(loss_names):
        top_scores_concat[d_name][l_name] = {}
        for nr in nr_rates:
            top_scores_concat[d_name][l_name][nr] = []
            for i_run in range(len(predictions[d_name][l_name][nr])):
                # Initial shape is (n_samples, n_classes)
                byrun_score = byrun_scores_ordered[d_name][l_name][nr][i_run]  # shape (n_samples, n_classes)
                # Top score across the classes (final shape (n_samples,))
                run_top_scores = np.max(byrun_score, axis=1)  # shape (n_samples,)
                top_scores_concat[d_name][l_name][nr].append(run_top_scores)
            # Concatenate the 5 runs (final shape (n_runs * n_samples,))
            top_scores_concat[d_name][l_name][nr] = np.concatenate(top_scores_concat[d_name][l_name][nr])


In [ ]:
# Max of mean score
top_scores_max_of_mean = {}
for d_name in dataset_names:
    loss_names = included_losses[d_name]
    top_scores_max_of_mean[d_name] = {}
    
    # Get the scores list for each loss and each nr
    for i_l, l_name in enumerate(loss_names):
        top_scores_max_of_mean[d_name][l_name] = {}
        for nr in nr_rates:
            # Compute the mean predictions across runs (final shape (n_samples, n_classes)), average can be done because samples are sorted according to index and thus correspond between runs
            mean_score = np.mean([run_scores for run_scores in byrun_scores_ordered[d_name][l_name][nr]], axis=0)  # shape (n_samples, n_classes)
            top_scores_max_of_mean[d_name][l_name][nr] = np.max(mean_score, axis=1)  # shape (n_samples,)

## Compute distances to theoretical value

In [ ]:
chosen_quantif_fn_name = "closest local max"
n_bins = 40
bin_edges = np.linspace(0, 1, n_bins + 1)  # n_bins bins from 0 to 1
print(bin_edges)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     
print(bin_edges.shape)

In [ ]:
# Get theoretical minimum for each loss and each noise rate

theoretical_mins = {}
for d_name in dataset_names:
    theoretical_mins[d_name] = {}
    loss_names = included_losses[d_name]
    for i_l, l_name in enumerate(loss_names):
        theoretical_mins[d_name][l_name] = {}
        for nr in nr_rates:
            # Get the theoretical minimum for the loss and noise rate
            K = target_K[d_name]
            eta = nr
            v_min_fn = loss_theo_min(l_name)
            v_min = v_min_fn(eta, K)
            
            # Get the theoretical minimum bin index
            theo_min = loss_theo_min(l_name)(nr, target_K[d_name])
            theoretical_mins[d_name][l_name][nr] = theo_min
print("Theoretical minimums:", theoretical_mins)

In [ ]:
# To compute distances between theoretical and score distribution
def get_distances(considered_scores, theoretical_mins, chosen_quantif_fn, bin_edges):
    distances = {}
    hist_lists = {}

    for l_name in considered_scores.keys():
        distances[l_name] = {}
        hist_lists[l_name] = {}
        for nr in considered_scores[l_name].keys():
            exp_scores = considered_scores[l_name][nr]
            hist, _ = np.histogram(exp_scores, bins=bin_edges)
            hist_lists[l_name][nr] = hist
            
            theo_min = theoretical_mins[l_name][nr]
            if theo_min is None:
                distance = None
            else:
                distance = chosen_quantif_fn(exp_bin_values=hist, test_theo=theo_min, bin_edges=bin_edges)
                print(f"{l_name} - nr {nr}")
                print(f"  Theoretical minimum: {theo_min:.4f}, associated bin: {np.searchsorted(bin_edges, theo_min) - 1} ({bin_edges[np.searchsorted(bin_edges, theo_min) - 1]} to {bin_edges[np.searchsorted(bin_edges, theo_min)]}), quantification: {distance}")
            distances[l_name][nr] = distance
    return distances, hist_lists

# Show distances in the form of matrices
def get_distance_matrix(distances, loss_names, nr_rates):
    """Get the distance matrix of a given distance dictionary. Loss names and nr rates are specified to have consistent line and columns order

    Args:
        distances (dict): Dictionary containing the distances for each dataset, loss, and noise rate.
        loss_names (list)
        nr_rates (list)

    Returns:
        np.array: Matrix of distances with shape (len(loss_names), len(nr_rates)), where rows correspond to loss names and columns correspond to noise rates.
    """
    
    # Get the distance matrix for each dataset
    distance_matrices = np.zeros((len(loss_names), len(nr_rates)))
    for i_l, l_name in enumerate(loss_names):
        for i_nr, nr in enumerate(nr_rates):
            distance_matrices[i_l, i_nr] = distances[l_name][nr]
    return distance_matrices

## Visualization

In [ ]:

vizualized_losses = [
    "CE_dropout", "CE_nodropout", "GCE_q0-4_nodropout", "GCE_q0-7_nodropout", "GCE_q1_nodropout",
    "balancedCE_dropout", "balancedCE_nodropout", "balancedGCE_q0-4_nodropout", "balancedGCE_q0-7_nodropout", "balancedGCE_q1_nodropout",
    ]
score_names = {"Max score concat": top_scores_concat, "Mean of max scores": top_scores_mean, "Max of mean score": top_scores_max_of_mean}

In [ ]:
hist_lists = {}
distance_lists = {}
for d_name in dataset_names:
    print(d_name)
    plt.figure(figsize=(30, 6))
    vmin=-0.8
    vmax=0.8
    vrange = max(abs(vmin), abs(vmax))
    for i_score, (score_name, exp_score) in enumerate(score_names.items()):
        print(f"Distance matrix for score type {score_name}:")
        exp_score_d = exp_score[d_name]
        distances, hist_list = get_distances(exp_score_d, theoretical_mins[d_name], compare_distance_functions[chosen_quantif_fn_name], bin_edges)
        
        # Gather histograms and scores
        hist_lists[(d_name, score_name)] = hist_list
        distance_lists[(d_name, score_name)] = distances

        loss_names = included_losses[d_name]
        loss_names = [l for l in loss_names if l in vizualized_losses]
        distance_matrix = get_distance_matrix(distances, loss_names, nr_rates)
        
        # Show the distance matrix as a heatmap, with fixed heat range between -1 and 1
        plt.subplot(1, 3, i_score + 1)
        sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap="coolwarm", xticklabels=nr_rates, yticklabels=loss_names, vmin=-vrange, vmax=vrange)
        plt.xlabel("Noise rate")
        plt.ylabel("Loss function")
        plt.title(f"{d_name} - {score_name} - {chosen_quantif_fn_name}\nMean abs error: {np.nanmean(np.abs(distance_matrix)):.4f}, Std abs error: {np.nanstd(np.abs(distance_matrix)):.4f}")
        
        # Gather error metrics
    # tmp_path = exp_root_path/f"distance_{d_name}_{score_names[score_name].replace(' ', '_')}_{chosen_quantif_fn_name.replace(' ', '_')}.json"
    plt.suptitle(f"Distance between theo minimum and exp scores mean for {d_name}")
    plt.show()

In [ ]:
# Visualize the distance matrices for each dataset and each score type, absolute value of the distance
for d_name in dataset_names:
    print(d_name)
    plt.figure(figsize=(30, 6))
    vmin=0
    vmax=0.8
    for i_score, (score_name, exp_score) in enumerate(score_names.items()):
        print(f"Distance matrix for score type {score_name}:")
        distances = distance_lists[(d_name, score_name)]


        loss_names = included_losses[d_name]
        loss_names = [l for l in loss_names if l in vizualized_losses]
        distance_matrix = np.abs(get_distance_matrix(distances, loss_names, nr_rates))
        
        # Show the distance matrix as a heatmap, with fixed heat range between -1 and 1
        plt.subplot(1, 3, i_score + 1)
        sns.heatmap(distance_matrix, annot=True, fmt=".2%", cmap="Reds", xticklabels=nr_rates, yticklabels=loss_names, vmin=vmin, vmax=vmax)
        plt.xlabel("Noise rate")
        plt.ylabel("Loss function")
        plt.title(f"{d_name} - {score_name} (abs) - {chosen_quantif_fn_name}\nMean distance: {np.nanmean(distance_matrix):.4f}, Std distance: {np.nanstd(distance_matrix):.4f}")
    plt.suptitle(f"Distance between theo minimum and exp scores mean for {d_name}")
    plt.show()

In [ ]:
# Final figure with one method for the histogram computation, no absolute
score_name = "Mean of max scores"
print(f"Visualizing histograms for score type {score_name}, with quantification method {chosen_quantif_fn_name}")
plt.figure(figsize=(35, 6))
for d_name in dataset_names:
    exp_score = score_names[score_name][d_name] # here we choose the top score concat for the histogram visualization
    distances, hist_list = get_distances(exp_score, theoretical_mins[d_name], compare_distance_functions[chosen_quantif_fn_name], bin_edges)

    loss_names = included_losses[d_name]
    loss_names = [l for l in loss_names if l in vizualized_losses]
    distance_matrix = get_distance_matrix(distance_lists[(d_name, score_name)], loss_names, nr_rates)
    
    # Show the distance matrix as a heatmap, with fixed heat range between -1 and 1
    plt.subplot(1, len(dataset_names), dataset_names.index(d_name) + 1)
    sns.heatmap(distance_matrix, annot=True, fmt=".2f", cmap="coolwarm", xticklabels=nr_rates, yticklabels=[loss_viz_names.get(ln, ln) for ln in loss_names], vmin=-0.8, vmax=0.8)
    plt.xlabel("Noise rate")
    plt.ylabel("Loss function")
    plt.title(f"{d_name}\nMean distance: {np.nanmean(distance_matrix):.4f}, Std distance: {np.nanstd(distance_matrix):.4f}")
plt.suptitle(f"Histograms of scores for {score_name}")
plt.show()

In [ ]:
# Visualization with mean of max histograms, each column a loss function, each row a nr rate

import matplotlib.lines as mpl
from matplotlib.patches import Rectangle

print(score_names.keys())
score_name = "Mean of max scores" # 'Max score concat', 'Mean of max scores', 'Max of mean score'
print(f"Visualizing histograms for score type {score_name}, with quantification method {chosen_quantif_fn_name}")
individual_title_fontsize = 14

publi_mode=True
publi_colors = "gray"
publi_theo_pt = ("red", "-")
publi_theo_line = ("black", "-")
publi_theo_pt_exp = ("orange", "-")
publi_orientation = "horizontal" # "horizontal" or "vertical"

for d_name in dataset_names:
    loss_names = included_losses[d_name]
    figsize = (4 * len(nr_rates), 2 * len(included_losses[d_name])) if publi_orientation == "vertical" else (2.5 * len(nr_rates), 4 * len(included_losses[d_name]))
    fig, axs = plt.subplots(len(included_losses[d_name]), len(nr_rates),  figsize=figsize, sharey=True)
    for i_l, l_name in enumerate(loss_names):
        if l_name in included_losses[d_name]:
            for i_nr, nr in enumerate(nr_rates):
                # Get histogram
                hist_values = hist_lists[(d_name, score_name)][l_name][nr]
                
                # Plot histogram
                bin_width = bin_edges[1] - bin_edges[0]
                if publi_orientation == "horizontal":
                    axs[included_losses[d_name].index(l_name), i_nr].barh(
                        bin_edges[:-1] + bin_width / 2, 
                        hist_values, 
                        height=bin_width, 
                        color=nr_colors[nr] if not publi_mode else publi_colors, 
                        alpha=0.5,
                        label="Score distribution"
                    )
                    axs[included_losses[d_name].index(l_name), i_nr].set_yticks(np.linspace(0, 1, 5))
                    axs[included_losses[d_name].index(l_name), i_nr].set_xlim(0, max(hist_values) * 1.1)
                    axs[included_losses[d_name].index(l_name), i_nr].set_ylim(0, 1.05)
                    axs[included_losses[d_name].index(l_name), i_nr].set_ylabel("Mean of max. score", fontsize=axis_fontsize)
                    axs[included_losses[d_name].index(l_name), i_nr].set_xlabel("Count", fontsize=axis_fontsize)
                    

                else:
                    axs[included_losses[d_name].index(l_name), i_nr].bar(
                        bin_edges[:-1] + bin_width / 2, 
                        hist_values, 
                        width=bin_width, 
                        color=nr_colors[nr] if not publi_mode else publi_colors, 
                        alpha=0.5,
                        label="Score distribution",
                    )
                    axs[included_losses[d_name].index(l_name), i_nr].set_xticks(np.linspace(0, 1, 5))
                    axs[included_losses[d_name].index(l_name), i_nr].set_ylim(0, max(hist_values) * 1.1)
                    axs[included_losses[d_name].index(l_name), i_nr].set_xlim(0, 1.05)
                    print(max(hist_values))

                    axs[included_losses[d_name].index(l_name), i_nr].set_xlabel("Mean of max. score", fontsize=axis_fontsize)
                    axs[included_losses[d_name].index(l_name), i_nr].set_ylabel("Count", fontsize=axis_fontsize)

            
                # Quantify distance with theory
                theo_min = theoretical_mins[d_name][l_name][nr]
                distance = distance_lists[(d_name, score_name)][l_name][nr]
                if theo_min is not None:
                    closest_local_max, _, _ = get_closest_local_max(exp_bin_values=hist_values, bin_edges=bin_edges, test_theo=theo_min)
                    local_max_high = bin_edges[closest_local_max + 1]
                    local_max_low = bin_edges[closest_local_max]
                    
                    ax = axs[included_losses[d_name].index(l_name), i_nr]

                    # Theoretical optimum
                    if publi_orientation == "vertical":
                        ax.axvline(
                            theo_min,
                            color=nr_colors[nr] if not publi_mode else publi_theo_pt[0],
                            linestyle="--" if not publi_mode else publi_theo_pt[1],
                            label="Theoretical optimum",
                            zorder=10
                        )

                        # Rectangle around the closest local-max bin
                        rect = Rectangle(
                            (local_max_low, 0),
                            local_max_high - local_max_low,
                            hist_values[closest_local_max] * 1.,
                            fill=False,
                            edgecolor=nr_colors[nr] if not publi_mode else publi_theo_pt_exp[0],
                            linewidth=2.5,
                            zorder=5,
                            label="Exp. closest local max. range"
                        )
                        ax.add_patch(rect)

                    else:
                        ax.axhline(
                            theo_min,
                            color=nr_colors[nr] if not publi_mode else publi_theo_pt[0],
                            linestyle="--" if not publi_mode else publi_theo_pt[1],
                            label="Theoretical optimum",
                            zorder=10
                        )

                        # Rectangle around the closest local-max bin
                        rect = Rectangle(
                            (0, local_max_low),
                            hist_values[closest_local_max] * 1.,
                            local_max_high - local_max_low,
                            fill=False,
                            edgecolor=nr_colors[nr] if not publi_mode else publi_theo_pt_exp[0],
                            linewidth=2.5,
                            zorder=5,
                            label="Exp. closest local max. range",
                        )
                        ax.add_patch(rect)

                avg_metric_info = f"\nAvg MCC test: {avg_metrics[d_name][nr][l_name][mode]['mcc ' + target_names[d_name]]:.2f} ± {std_metrics[d_name][nr][l_name][mode]['mcc ' + target_names[d_name]]:.2f}" if avg_metrics[d_name][nr][l_name][mode]['mcc ' + target_names[d_name]] is not None else "N/A"
                loss_str = loss_viz_names.get(l_name, l_name)
                dist_str = f"{distance:.2f}" if distance is not None else f"{loss_viz_names.get(l_name, l_name)}" + "N/A"
                if publi_mode:
                    title_full_str = f"Noise rate {nr}" +(f", loss {loss_str}" if publi_orientation == "vertical" else f"")
                else:
                    title_full_str = f"Noise rate {nr}, loss {loss_str}, distance {dist_str}" + avg_metric_info
                if i_nr == 0:
                    axs[included_losses[d_name].index(l_name), 0].set_title(title_full_str, fontsize=individual_title_fontsize)
                else:
                    axs[included_losses[d_name].index(l_name), i_nr].set_title(title_full_str, fontsize=individual_title_fontsize)
    #-- Legend and suptitle
    handles = axs[0, 0].get_legend_handles_labels()[0]  # Get the handles from the first subplot
    print([type(h) for h in handles])
    # handles.append(mpl.Line2D([], [], color=publi_theo_pt[0], linestyle=publi_theo_pt[1], label="Theoretical optimum"))
    # handles.append(mpl.Line2D([], [], color=publi_theo_pt_exp[0], alpha=0.4, label="Exp. closest local max. range"))
    axs[0, len(nr_rates) - 1].legend(handles=handles, loc='upper left', bbox_to_anchor=(1.0, 1.05)) # Legend on the right of the first row of subplots
    # plt.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=len(handles), fontsize=individual_title_fontsize) # Legend below the last row of subplots
    plt.suptitle(f"{d_name} - {mode}", fontsize=individual_title_fontsize)
    
    plt.tight_layout()
    # Save svg figure
    plt.savefig(exp_root_path/exp_roots[d_name][0]/f"histograms_{d_name.replace(' ', '')}_{score_name.replace(' ', '_')}_{chosen_quantif_fn_name.replace(' ', '_')}_nbins{n_bins}_new.svg", format='svg')
    plt.show()

# 8s

In [ ]:
# Visualization with matrix of bins
from matplotlib.colors import ListedColormap, LogNorm
from matplotlib.patches import Patch, Rectangle
print(f"Number of bins: {n_bins}, bin edges: {bin_edges}")
publi_mode=True

score_name = "Mean of max scores" # 'Max score concat', 'Mean of max scores', 'Max of mean score'
for d_name in dataset_names:
    is_PTB = d_name == "PTB"
    loss_names = included_losses[d_name]
    fig, axs = plt.subplots(
        1,
        len(included_losses[d_name]) + 1,
        figsize=(4 * len(included_losses[d_name]), 4),
        gridspec_kw={'width_ratios': [1] * len(included_losses[d_name]) + [0.04]}
    )

    for i_l, l_name in enumerate(included_losses[d_name]):
        # Get histogram
        hist_values = hist_lists[(d_name, score_name)][l_name][nr]

        # Gather the bins of each noise rate for the current loss
        bins_matrix = np.array([
            hist_lists[(d_name, score_name)][l_name][nr]
            for nr in nr_rates
            if not(nr == 0.6 and is_PTB)
        ])
        bins_matrix = bins_matrix.T  # shape (n_bins, n_nr)

        # Show the bins matrix as a heatmap
        eps = 100
        sns.heatmap(
            bins_matrix,
            annot=False,
            cmap="Greys",
            xticklabels=[
                nr for nr in nr_rates
                if not(nr == 0.6 and is_PTB)
            ],
            ax=axs[i_l],
            cbar_kws={"label": "Count"},
            cbar=(i_l == len(included_losses[d_name]) - 1),
            cbar_ax=axs[-1],
            zorder=0,
        )

        step = 10

        # Show theoretical values on the heatmap
        theo_indices = [
            theoretical_mins[d_name][l_name][nr] * (len(bin_edges) - 1)
            for nr in nr_rates
            if not(nr == 0.6 and is_PTB)
        ]


        sns.lineplot(
            x=[i + 0.5 for i in range(len(theo_indices))],
            y=theo_indices,
            color=publi_theo_line[0],
            linestyle=publi_theo_line[1],
            marker="o",
            markersize=3,
            markeredgecolor=publi_theo_line[0],
            markerfacecolor=publi_theo_line[0],
            label="Theo. optimum trend",
            ax=axs[i_l],
            legend=False,
            zorder = 30
        )

        sns.scatterplot(
            x=[i + 0.5 for i in range(len(theo_indices))],
            y=theo_indices,
            color=publi_theo_pt[0],
            marker="o",
            s=100,
            label="Theo. optimum",
            ax=axs[i_l],
            legend=False,
            zorder = 20
        )

        # ---------------------------------------------------------
        # Show closest local maximum by drawing a rectangle
        # around the corresponding bin
        # ---------------------------------------------------------
        local_maxs = []

        for i_nr, nr in enumerate(nr_rates):
            if not(nr == 0.6 and is_PTB):
                closest_local_max, _, _ = get_closest_local_max(
                    exp_bin_values=hist_lists[(d_name, score_name)][l_name][nr],
                    bin_edges=bin_edges,
                    test_theo=theoretical_mins[d_name][l_name][nr]
                )
                local_maxs.append(closest_local_max)

        print([
            (theoretical_mins[d_name][l_name][nr], theo)
            for nr, theo in zip(
                [nr for nr in nr_rates if not(nr == 0.6 and is_PTB)],
                theo_indices
            )
        ])
        print([bin_edges[int(lm):int(lm) + 2] for lm in local_maxs])
        print("Differences = ", [compare_value_to_bin(theoretical_mins[d_name][l_name][nr], bin_edges[int(local_maxs[i_nr]):int(local_maxs[i_nr]) + 2]) for i_nr, nr in enumerate([nr for nr in nr_rates if not(nr == 0.6 and is_PTB)])])
        print("Differences bins = ", [compare_value_to_bin(theo_indices[i_nr], [int(local_maxs[i_nr]), int(local_maxs[i_nr]) + 1]) for i_nr, nr in enumerate([nr for nr in nr_rates if not(nr == 0.6 and is_PTB)])])

        # Draw a rectangle around each selected bin
        for i_nr, lm in enumerate(local_maxs):
            rect = Rectangle(
                (i_nr, lm),                         # bottom-left corner
                1,                                   # width
                1,                                   # height
                fill=False,
                edgecolor=publi_theo_pt_exp[0],
                linewidth=2.5,
                zorder=10
            )
            axs[i_l].add_patch(rect)

        # Optional: keep the trend line connecting the local maxima
        sns.lineplot(
            x=[i + 0.5 for i in range(len(local_maxs))],
            y=[lm + 0.5 for lm in local_maxs],
            linestyle="--",
            marker="o",
            markersize=3,
            markeredgecolor=publi_theo_line[0],
            markerfacecolor=publi_theo_line[0],
            color=publi_theo_line[0],
            label="Exp. closest local max. trend",
            ax=axs[i_l],
            legend=False,
            zorder = 25
        )

        axs[i_l].axes.invert_yaxis()

        axs[i_l].set_yticks(
            np.concatenate([
                np.arange(0, len(bin_edges) - 1, step),
                [len(bin_edges) - 1]
            ])
        )
        axs[i_l].set_yticklabels(
            np.concatenate([
                np.round(bin_edges[:-1][::step], 2),
                [np.round(bin_edges[-1], 2)]
            ]),
            rotation=0
        )

        axs[i_l].set_ylabel(score_name, fontsize=axis_fontsize)
        axs[i_l].set_xlabel("Noise rate", fontsize=axis_fontsize)
        axs[i_l].set_title(
            f"{loss_viz_names.get(l_name, l_name)}",
            fontsize=title_fontsize
        )

    # Put legend below the subplots
    handles, labels = axs[i_l].get_legend_handles_labels()

    handles.append(
        Rectangle(
            (0, 0),
            1,
            1,
            fill=False,
            edgecolor=publi_theo_pt_exp[0],
            linewidth=1.0
        )
    )
    labels.append("Exp. closest local max. bin")

    fig.legend(
        handles,
        labels,
        loc="lower center",
        ncol=2,
        fontsize=legend_fontsize,
        bbox_to_anchor=(0.5, -0.12)
    )

    if not publi_mode:
        plt.suptitle(
            f"{d_name} - {mode} - Histogram bins",
            fontsize=individual_title_fontsize
        )

    plt.tight_layout()

    plt.savefig(
        exp_root_path / exp_roots[d_name][0] /
        f"histogram_bins_{d_name.replace(' ', '')}_"
        f"{score_name.replace(' ', '_')}_"
        f"{chosen_quantif_fn_name.replace(' ', '_')}_"
        f"nbins{n_bins}_new.svg",
        format='svg'
    )

    plt.show()


## Analyze bin size and method impacts

In [ ]:
# Save the distance matrices for each dataset and each score type
save_json = {score_name: {} for score_name in score_names}
for d_name in dataset_names:
    for i_score, (score_name, exp_score) in enumerate(score_names.items()):
        print(exp_score.keys())
        print(theoretical_mins[d_name].keys())
        distances, hist_list = get_distances(exp_score[d_name], theoretical_mins[d_name], compare_distance_functions[chosen_quantif_fn_name], bin_edges)
        loss_names = included_losses[d_name]
        loss_names = [l for l in loss_names if l in vizualized_losses]
        distance_matrix = get_distance_matrix(distances, loss_names, nr_rates)
        save_json[score_name][d_name] = {
            "distances": distances,
            "distance_matrix": distance_matrix.tolist(),
            "loss_names": loss_names,
            "nr_rates": nr_rates,
            "mean_abs_error": np.nanmean(np.abs(distance_matrix)),
            "std_abs_error": np.nanstd(np.abs(distance_matrix))
        }
with (exp_root_path/f"distance_matrices_{n_bins}_{chosen_quantif_fn_name.replace(' ', '_')}.json").open("w") as f:
    json.dump(save_json, f, indent=4)

In [ ]:
errors_dict = {
    20: {
        "closest global max": "distance_matrices_20_closest_global_max.json",
        "closest local max": "distance_matrices_20_closest_local_max.json",
    },
    30: {
        "closest global max": "distance_matrices_30_closest_global_max.json",
        "closest local max": "distance_matrices_30_closest_local_max.json",
    },
    40: {
        "closest global max": "distance_matrices_40_closest_global_max.json",
        "closest local max": "distance_matrices_40_closest_local_max.json",
    },
    50: {
        "closest global max": "distance_matrices_50_closest_global_max.json",
        "closest local max": "distance_matrices_50_closest_local_max.json",
    }
}
n_bins_list = sorted(list(errors_dict.keys()))
chosen_fn_list = sorted(errors_dict[n_bins_list[0]].keys(), reverse=True)

In [ ]:
for d_name in dataset_names:
    plt.figure(figsize=(8, 4))
    for score_name in score_names:
        new_score_name = True
        color = "grey"
        for i_fn, chosen_fn in enumerate(chosen_fn_list):
            mean_by_bin = []
            std_by_bin = []
            for i_n, n_bins in enumerate(n_bins_list):
                with (exp_root_path/errors_dict[n_bins][chosen_fn]).open("r") as f:
                    data = json.load(f)
                if score_name not in data:
                    print(f"Score name {score_name} not found in {errors_dict[n_bins][chosen_fn]}")
                    continue
                mean_abs_error = data[score_name][d_name]["mean_abs_error"]
                std_abs_error = data[score_name][d_name]["std_abs_error"]
                mean_by_bin.append(mean_abs_error)
                std_by_bin.append(std_abs_error)
            if i_fn == 0 and new_score_name:
                fig = plt.plot(n_bins_list, mean_by_bin, "x--", label=f"{chosen_fn} - {score_name}")
                color = fig[0].get_color()
                new_score_name = False
            else:
                plt.plot(n_bins_list, mean_by_bin, "x--" if i_fn == 0 else "x-", label=f"{chosen_fn} - {score_name}", color=color)
            # plt.errorbar(n_bins_list, mean_by_bin, yerr=std_by_bin, fmt='o', color=color, capsize=5, alpha=0.5)
    plt.ylabel("Mean absolute error")
    plt.title(f"Mean absolute error vs number of bins for {d_name} - {score_name}")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

for d_name in dataset_names:
    rows = []
    plt.figure(figsize=(15, 5))
    ax = None
    for chosen_fn in chosen_fn_list:
        ax = plt.subplot(1, len(chosen_fn_list), list(chosen_fn_list).index(chosen_fn) + 1, sharey=ax if ax is not None else None)
        for score_name in score_names:
            for n_bins in n_bins_list:

                with (exp_root_path / errors_dict[n_bins][chosen_fn]).open("r") as f:
                    data = json.load(f)

                score_list = (
                    np.array(data[score_name][d_name]["distance_matrix"])
                    .flatten()
                    .tolist()
                )

                rows.extend(
                    {
                        "n_bins": n_bins,
                        "score_strategy": score_name,
                        "error_with_theo": score,
                        "chosen_fn": chosen_fn,
                    }
                    for score in score_list
                )

        df = pd.DataFrame(rows)


        sns.boxplot(
            data=df,
            x="n_bins",
            y="error_with_theo",
            hue="score_strategy",
            showfliers=False,
        )
        plt.axhline(0, color="black", linestyle="--", label="Zero line")

        plt.xlabel("Number of bins")
        plt.ylabel("Error with theory (all losses and nr mixed)")
        plt.title(d_name + f" - {chosen_fn}")
    plt.legend(title="Score strategy", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()